In [1]:
from src.transformer.transformer_architecture import LitKeypointTransformer
import pickle
from sklearn.metrics import classification_report
from tqdm import tqdm
import torch

In [2]:
model = LitKeypointTransformer.load_from_checkpoint("checkpoints/keypoint_transformer2.ckpt")
model.eval()

LitKeypointTransformer(
  (model): KeypointTransformer(
    (input_proj): Sequential(
      (0): Linear(in_features=126, out_features=132, bias=True)
    )
    (pos_encode): SinusoidalPositionalEncoding()
    (layers): ModuleList(
      (0-5): 6 x EncodingLayer(
        (attention): MultiHeadAttention(
          (w_q): Linear(in_features=132, out_features=132, bias=True)
          (w_k): Linear(in_features=132, out_features=132, bias=True)
          (w_v): Linear(in_features=132, out_features=132, bias=True)
          (output): Linear(in_features=132, out_features=132, bias=True)
        )
        (norm1): LayerNorm((132,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((132,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
        (ffn): Sequential(
          (0): Linear(in_features=132, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_f

In [3]:
with open("data/dataset.pkl", "rb") as f:
    ds = pickle.load(f)

In [4]:
# Code from gym.ipynb
N_F = 5649
TEST_F = int(0.1*N_F)
TRAIN_VAL_F = N_F - TEST_F

test_ds = ds[TRAIN_VAL_F:]

In [5]:
all_labels = []
all_preds = []

for i in tqdm(range(len(test_ds))):    
    keypoint_sequence, label = test_ds[i]

    x = torch.tensor(keypoint_sequence, dtype=torch.float32).unsqueeze(0)  # (1, seq_len, 126)

    # prediction
    with torch.no_grad():
        logits = model(x)
        predicted_idx = torch.argmax(logits, dim=-1).item()

    all_labels.append(label)
    all_preds.append(predicted_idx)

100%|██████████| 564/564 [00:01<00:00, 338.68it/s]


In [6]:
gesture_labels = ['Non-gesture', 'Pointing with one finger', 'Pointing with two fingers', 'Click with one finger', 
          'Click with two fingers', 'Throw up', 'Throw down', 'Throw left', 'Throw right', 'Open twice', 'Double click with one finger',
          'Double click with two fingers', 'Zoom in', 'Zoom out']

In [7]:
class_report = classification_report(all_labels, all_preds)
print(class_report)

              precision    recall  f1-score   support

           0       0.89      0.90      0.89       148
           1       0.96      0.92      0.94       101
           2       0.95      0.92      0.93        98
           3       0.50      0.68      0.58        19
           4       0.29      0.60      0.39        20
           5       0.65      0.85      0.74        20
           6       1.00      0.60      0.75        20
           7       0.71      0.79      0.75        19
           8       0.78      0.70      0.74        20
           9       0.56      0.25      0.34        20
          10       0.67      0.30      0.41        20
          11       0.31      0.40      0.35        20
          12       0.67      0.90      0.77        20
          13       0.14      0.05      0.08        19

    accuracy                           0.77       564
   macro avg       0.65      0.63      0.62       564
weighted avg       0.79      0.77      0.77       564

